# 01 — Data cleaning and alignment

**Dataset:** [S&P 500 with Financial News Headlines (2008–2024)](https://www.kaggle.com/datasets/dyutidasmahaptra/s-and-p-500-with-financial-news-headlines-20082024)
(Kaggle, Dyuti Dasmahapatra). Each row is a real financial news headline with its
publication date and the S&P 500 closing price (`CP`) on that date.

This notebook cleans the raw file and documents every alignment issue that
affects the analysis downstream: duplicates, sparse early coverage, and —
most importantly — the fact that prices only exist on dates that have
headlines, which makes a naive "next row = next day" assumption wrong.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 100)

from src.data import download_raw, load_headlines, build_price_frame, coverage_report

raw_path = download_raw()
raw = pd.read_csv(raw_path)
headlines = load_headlines(raw_path)
print(f"raw rows:   {len(raw):,}")
print(f"clean rows: {len(headlines):,}  (dropped {len(raw) - len(headlines):,} exact date+headline duplicates)")
headlines.head(8)

raw rows:   19,127
clean rows: 18,153  (dropped 974 exact date+headline duplicates)


,headline,date,close
0,"JPMorgan Predicts 2008 Will Be ""Nothing But Net""",2008-01-02,1447.16
1,Dow Tallies Biggest First-session-of-year Point Drop Ever,2008-01-02,1447.16
2,2008 predictions for the S&P 500,2008-01-02,1447.16
3,"U.S. Stocks Higher After Economic Data, Monsanto Outlook",2008-01-03,1447.16
4,U.S. Stocks Climb As Hopes Increase For More Fed Moves,2008-01-07,1416.18
5,How Investing in Intangibles -- Like Employee Satisfaction -- Translates into Financial Returns,2008-01-09,1409.13
6,Head And Shoulders Top Bodes Ill For Bulls,2008-01-09,1409.13
7,U.S. Stocks Zigzag Higher As Bernanke Speech Sinks In,2008-01-10,1420.33


## Coverage by year

Coverage is thin before 2011 (~36–52% of trading days) and near-complete after.
Every downstream result is therefore also checked on the dense 2011+ subsample.

In [2]:
coverage_report(headlines)

,headlines,days,approx_coverage
date,,,
2008,141,106,0.42
2009,113,90,0.36
2010,209,132,0.52
2011,547,216,0.86
2012,722,241,0.96
2013,775,241,0.96
2014,856,242,0.96
2015,818,243,0.96
2016,801,240,0.95


## The alignment problem

The dataset has no price on a trading day with no headlines. So "the next row"
after day *t* is sometimes several sessions later, and treating that as a
next-day move would silently inflate "next-day" returns (e.g. the row after
2008-10-13 is 2008-10-15 — a two-session move of −9.5%).

`build_price_frame` builds an NYSE session calendar (weekends, market holidays,
and unscheduled closures like Hurricane Sandy 2012) and marks `next_day_ok = True`
only when the next observed date is the *immediately following* trading session.

In [3]:
daily = build_price_frame(headlines)
print(f"daily rows: {len(daily):,}")
print(f"genuine next-trading-day pairs: {daily.next_day_ok.mean():.1%}")
print()
print("share of valid next-day pairs by year:")
print(daily.groupby(daily.date.dt.year).next_day_ok.mean().round(2).to_string())

daily rows: 3,507
genuine next-trading-day pairs: 91.5%

share of valid next-day pairs by year:
date
2008    0.39
2009    0.36
2010    0.53
2011    0.88
2012    0.96
2013    0.96
2014    0.96
2015    0.96
2016    0.96
2017    0.92
2018    0.97
2019    0.98
2020    0.98
2021    0.99
2022    1.00
2023    1.00
2024    0.98


In [4]:
# The case that motivated the calendar: 2008-10-14 is missing, so the pair
# starting 2008-10-13 spans two sessions and is excluded. 2020-03-13 -> 03-16
# (a genuine Friday-to-Monday crash of -12%) is correctly kept.
daily[daily.date.isin(pd.to_datetime(["2008-10-13", "2020-03-13"]))][
    ["date", "close", "next_ret", "gap_to_next", "next_day_ok"]
]

,date,close,next_ret,gap_to_next,next_day_ok
76,2008-10-13,1003.35,-0.095191,2.0,False
2519,2020-03-13,2711.02,-0.119841,3.0,True


## Price integrity spot checks

The `CP` column matches official S&P 500 closes exactly on dates picked across
the sample, including turning points where an error would be most visible.

In [5]:
checks = {"2020-03-23": 2237.40, "2008-09-29": 1106.42, "2013-12-31": 1848.36, "2022-01-03": 4796.56}
for dt, expected in checks.items():
    actual = daily.loc[daily.date == dt, "close"].iloc[0]
    print(f"{dt}: dataset {actual:>8.2f}   official {expected:>8.2f}   {'OK' if abs(actual-expected) < 0.01 else 'MISMATCH'}")

2020-03-23: dataset  2237.40   official  2237.40   OK
2008-09-29: dataset  1106.42   official  1106.42   OK
2013-12-31: dataset  1848.36   official  1848.36   OK
2022-01-03: dataset  4796.56   official  4796.56   OK


## What carries forward

- 18,153 clean (date, headline) rows over 3,507 trading dates, 2008-01-02 → 2024-03-04
- 91.5% of dates form a genuine next-trading-day pair (`next_day_ok`); only those
  are used for anything labelled "next-day"
- Dates are day-granular: we cannot tell whether a headline appeared before or
  after that day's close. This shapes the whole design — see notebook 03.